In [56]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import hashlib
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def afficher_planche_contact(dossier_classe, nb_colonnes=8, nb_lignes=8):
    fichiers = sorted(os.listdir(dossier_classe))
    figure, axes = plt.subplots(nb_lignes, nb_colonnes, figsize=(16, 16))

    index = 0
    for ligne in range(nb_lignes):
        for colonne in range(nb_colonnes):
            axe = axes[ligne][colonne]
            if index < len(fichiers):
                chemin = os.path.join(dossier_classe, fichiers[index])
                try:
                    image = Image.open(chemin)
                    axe.imshow(image)
                    axe.set_title(fichiers[index], fontsize=6)
                except Exception:
                    pass
            axe.axis("off")
            index += 1

    plt.tight_layout()
    plt.show()

# exemple : afficher_planche_contact("atelier_prepa_donnees_images/data/raw/glass")

# partie 1-Exploration du dataset

In [42]:


dossier_raw = "../data/raw"

def explorer_dataset(dossier_raw):
    lignes = []
    classes = sorted(os.listdir(dossier_raw))

    for classe in classes:
        dossier_classe = os.path.join(dossier_raw, classe)
        noms_fichiers = sorted(os.listdir(dossier_classe))

        for nom_fichier in noms_fichiers:
            chemin = os.path.join(dossier_classe, nom_fichier)
            taille_octets = os.path.getsize(chemin)

            ligne = {
                "nom": nom_fichier,
                "classe": classe,
                "chemin": chemin,
                "format": None,
                "mode": None,
                "largeur": None,
                "hauteur": None,
                "ecart_type": None,
                "nb_canaux": None,
                "taille_octets": taille_octets,
                "corrompue": False
            }

            try:
                image = Image.open(chemin)
                image.load()

                info_format = image.format
                info_mode = image.mode
                info_largeur = image.width
                info_hauteur = image.height

                if info_mode == "L":
                    nb_canaux = 1
                elif info_mode == "RGB":
                    nb_canaux = 3
                elif info_mode == "RGBA":
                    nb_canaux = 4
                else:
                    nb_canaux = len(image.getbands())

                tableau = np.array(image)
                ecart_type = round(float(tableau.std()), 2)

                ligne["format"] = info_format
                ligne["mode"] = info_mode
                ligne["largeur"] = info_largeur
                ligne["hauteur"] = info_hauteur
                ligne["nb_canaux"] = nb_canaux
                ligne["ecart_type"] = ecart_type

            except Exception as erreur:
                ligne["corrompue"] = True

            lignes.append(ligne)

    return lignes


lignes = explorer_dataset(dossier_raw)
print("Nombre total d'images explorees :", len(lignes))
print("Cles disponibles :", lignes[0].keys())

Nombre total d'images explorees : 1032
Cles disponibles : dict_keys(['nom', 'classe', 'chemin', 'format', 'mode', 'largeur', 'hauteur', 'ecart_type', 'nb_canaux', 'taille_octets', 'corrompue'])


# Partie 2 – Détecter les images corrompues

In [43]:
def est_corrompue(chemin_image):
    """Retourne True si l'image ne peut pas etre ouverte/lue correctement."""
    try:
        image = Image.open(chemin_image)
        image.load()
        return False
    except Exception:
        return True


dossier_raw = "../data/raw"
classes = sorted(os.listdir(dossier_raw))
images_corrompues = []

for classe in classes:
    dossier_classe = os.path.join(dossier_raw, classe)
    for nom_fichier in sorted(os.listdir(dossier_classe)):
        chemin = os.path.join(dossier_classe, nom_fichier)
        if est_corrompue(chemin):
            images_corrompues.append(chemin)

print("Nombre d'images corrompues :", len(images_corrompues))

Nombre d'images corrompues : 6


# Partie 3 – Détecter les images vides

In [44]:
def est_vide(chemin_image, seuil_ecart_type=5.0):
    """
    Retourne True si l'image est consideree comme vide :
    entierement noire, entierement blanche, ou tres peu de variation.
    """
    try:
        image = Image.open(chemin_image)
        image.load()
    except Exception:
        return False   # une image corrompue est traitee a part (Partie 2)

    tableau = np.array(image)
    moyenne = tableau.mean()
    ecart_type = tableau.std()

    if ecart_type < seuil_ecart_type:
        return True
    if moyenne < 2:
        return True
    if moyenne > 253:
        return True
    return False


# on enregistre le resultat DANS la liste "lignes" pour le reutiliser plus tard
for ligne in lignes:
    if ligne["corrompue"]:
        ligne["vide"] = False
    else:
        ligne["vide"] = est_vide(ligne["chemin"])

images_vides = []
for ligne in lignes:
    if ligne["vide"]:
        images_vides.append(ligne["chemin"])

print("Nombre d'images quasi vides :", len(images_vides))
for chemin in images_vides:
    print("  -", chemin)

Nombre d'images quasi vides : 2
  - ../data/raw\cardboard\image-blanche-512x384.jpg
  - ../data/raw\metal\image-blanche-512x384.jpg


# Partie 4 – Détecter les différences de résolution

1) Déterminer la résolution minimale, la résolution maximale, les résolutions les plus
fréquentes et le nombre d'images par résolution. 
2) On décide qu'une image doit avoir au minimum 64 × 64 pixels. Identifier toutes les images
ne respectant pas cette contrainte.

In [45]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))
compteur_resolutions = {}
for ligne in lignes_valides:
    resolution = (ligne["largeur"], ligne["hauteur"])
    if resolution in compteur_resolutions:
        compteur_resolutions[resolution] += 1
    else:
        compteur_resolutions[resolution] = 1

toutes_largeurs = []
toutes_hauteurs = []
for ligne in lignes_valides:
    toutes_largeurs.append(ligne["largeur"])
    toutes_hauteurs.append(ligne["hauteur"])

print("Resolution minimale :", min(toutes_largeurs), "x", min(toutes_hauteurs))
print("Resolution maximale :", max(toutes_largeurs), "x", max(toutes_hauteurs))

resolutions_triees = sorted(compteur_resolutions.items(), key=lambda item: item[1], reverse=True)
print("Resolutions les plus frequentes :")
for resolution, nombre in resolutions_triees:
    print("  ", resolution, "->", nombre, "images")

images_trop_petites = []
for ligne in lignes_valides:
    if ligne["largeur"] < 64 or ligne["hauteur"] < 64:
        images_trop_petites.append(ligne["chemin"])

print("Nombre d'images trop petites (< 64x64) :", len(images_trop_petites))
for chemin in images_trop_petites:
    print("  -", chemin)

Nombre d'images valides (hors corrompues) : 1026
Resolution minimale : 32 x 32
Resolution maximale : 512 x 384
Resolutions les plus frequentes :
   (512, 384) -> 1013 images
   (32, 32) -> 5 images
   (48, 32) -> 4 images
   (40, 40) -> 4 images
Nombre d'images trop petites (< 64x64) : 13
  - ../data/raw\cardboard\cardboard117.jpg
  - ../data/raw\cardboard\cardboard22.jpg
  - ../data/raw\cardboard\cardboard70.jpg
  - ../data/raw\glass\glass100.jpg
  - ../data/raw\glass\glass15.jpg
  - ../data/raw\glass\glass21.jpg
  - ../data/raw\glass\glass23.jpg
  - ../data/raw\metal\metal121.jpg
  - ../data/raw\metal\metal2.jpg
  - ../data/raw\metal\metal26.jpg
  - ../data/raw\paper\paper10.jpg
  - ../data/raw\paper\paper54.jpg
  - ../data/raw\paper\paper64.jpg


# Partie 5 – Détecter les différents canaux

In [46]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))
compteur_canaux = {}
for ligne in lignes_valides:
    nb = ligne["nb_canaux"]
    if nb in compteur_canaux:
        compteur_canaux[nb] += 1
    else:
        compteur_canaux[nb] = 1

print("Repartition par nombre de canaux :", compteur_canaux)

Nombre d'images valides (hors corrompues) : 1026
Repartition par nombre de canaux : {3: 1006, 4: 18, 1: 2}


# Partie 6 – Détecter les doublons

In [47]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))


def calculer_empreinte_pixels(chemin_image, taille=(16, 16)):
    """
    Empreinte basee sur le contenu VISUEL de l'image (average-hash simplifie).
    Deux fichiers differents (format/taille differents) representant
    la meme image donneront la meme empreinte.

    LIMITE CONNUE : sur une image de couleur parfaitement uniforme
    (ecart-type nul), cette methode ne fonctionne pas bien -> voir Partie 7.
    """
    image = Image.open(chemin_image).convert("L")
    image = image.resize(taille)
    tableau = list(image.getdata())
    moyenne = sum(tableau) / len(tableau)

    bits = ""
    for valeur in tableau:
        if valeur > moyenne:
            bits += "1"
        else:
            bits += "0"
    return bits


empreintes_vues = {}
doublons_visuels = []
for ligne in lignes_valides:
    empreinte = calculer_empreinte_pixels(ligne["chemin"])
    if empreinte in empreintes_vues:
        doublons_visuels.append((empreintes_vues[empreinte], ligne["chemin"]))
    else:
        empreintes_vues[empreinte] = ligne["chemin"]

print("Nombre de doublons visuels detectes :", len(doublons_visuels))
for original, doublon in doublons_visuels:
    print("  ", original, "<->", doublon)

Nombre d'images valides (hors corrompues) : 1026


C:\Users\DELL\AppData\Local\Temp\ipykernel_18072\2690612668.py:20: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  tableau = list(image.getdata())


Nombre de doublons visuels detectes : 21
   ../data/raw\cardboard\cardboard4.jpg <-> ../data/raw\cardboard\cardboard4g.png
   ../data/raw\cardboard\cardboard85.jpg <-> ../data/raw\cardboard\cardboard85g.png
   ../data/raw\glass\glass75.jpg <-> ../data/raw\glass\glass75g.png
   ../data/raw\cardboard\image-violet-512x384.gif <-> ../data/raw\glass\image-noire-512x384.png
   ../data/raw\cardboard\image-violet-512x384.gif <-> ../data/raw\glass\image-violet-512x384.gif
   ../data/raw\cardboard\image-blanche-512x384.jpg <-> ../data/raw\metal\image-blanche-512x384.jpg
   ../data/raw\cardboard\image-violet-512x384.gif <-> ../data/raw\metal\image-noire-512x384.png
   ../data/raw\metal\metal125.jpg <-> ../data/raw\metal\metal125po.jpg
   ../data/raw\metal\metal13.jpg <-> ../data/raw\metal\metal13z.jpg
   ../data/raw\metal\metal35.jpg <-> ../data/raw\metal\metal35rt.jpg
   ../data/raw\metal\metal62.jpg <-> ../data/raw\metal\metal62rt.jpg
   ../data/raw\glass\glass115.jpg <-> ../data/raw\metal\meta

# Partie 7 : Détecter les images mal classées

In [48]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))




# on croise les doublons visuels avec la classe de chaque fichier ;
# on exclut les images "vides" car leur empreinte n'est pas fiable (voir Partie 6)
doublons_inter_classes = []

for original, doublon in doublons_visuels:
    classe_originale = None
    classe_doublon = None
    original_est_vide = False
    doublon_est_vide = False

    for ligne in lignes_valides:
        if ligne["chemin"] == original:
            classe_originale = ligne["classe"]
            original_est_vide = ligne["vide"]
        if ligne["chemin"] == doublon:
            classe_doublon = ligne["classe"]
            doublon_est_vide = ligne["vide"]

    if original_est_vide or doublon_est_vide:
        continue   # on ignore les faux positifs lies aux images uniformes

    if classe_originale != classe_doublon:
        doublons_inter_classes.append((original, classe_originale, doublon, classe_doublon))

print("Doublons entre classes differentes (forte suspicion de mauvais classement) :")
for original, classe_o, doublon, classe_d in doublons_inter_classes:
    print("  ", original, "(", classe_o, ")  <->  ", doublon, "(", classe_d, ")")

Nombre d'images valides (hors corrompues) : 1026
Doublons entre classes differentes (forte suspicion de mauvais classement) :
   ../data/raw\cardboard\image-violet-512x384.gif ( cardboard )  <->   ../data/raw\glass\image-noire-512x384.png ( glass )
   ../data/raw\cardboard\image-violet-512x384.gif ( cardboard )  <->   ../data/raw\glass\image-violet-512x384.gif ( glass )
   ../data/raw\cardboard\image-violet-512x384.gif ( cardboard )  <->   ../data/raw\metal\image-noire-512x384.png ( metal )
   ../data/raw\glass\glass115.jpg ( glass )  <->   ../data/raw\metal\metal91.jpg ( metal )
   ../data/raw\glass\glass176.jpg ( glass )  <->   ../data/raw\plastic\plastic152.jpg ( plastic )


# Partie 8 : Analyser le déséquilibre des classes

In [49]:
lignes_valides = []
for ligne in lignes:
    if not ligne["corrompue"]:
        lignes_valides.append(ligne)

print("Nombre d'images valides (hors corrompues) :", len(lignes_valides))

compteur_classes = {}
for ligne in lignes_valides:
    classe = ligne["classe"]
    if classe in compteur_classes:
        compteur_classes[classe] += 1
    else:
        compteur_classes[classe] = 1

print("Repartition par classe :")
for classe, nombre in sorted(compteur_classes.items()):
    print("  ", classe, "->", nombre)

Nombre d'images valides (hors corrompues) : 1026
Repartition par classe :
   cardboard -> 168
   glass -> 187
   metal -> 148
   paper -> 251
   plastic -> 223
   trash -> 49


****images_a_nettoyer****

In [50]:
chemins_a_exclure = set()

for ligne in lignes_valides:
    if ligne["vide"]:
        chemins_a_exclure.add(ligne["chemin"])
    if ligne["largeur"] < 64 or ligne["hauteur"] < 64:
        chemins_a_exclure.add(ligne["chemin"])

for original, doublon in doublons_visuels:
    # on garde "original" (le premier rencontre en parcourant le dossier), on exclut le doublon
    chemins_a_exclure.add(doublon)

# image mal classee confirmee en Partie 7 (goulot de bouteille en verre range dans metal/)
chemin_metal91 = os.path.join(dossier_raw, "metal", "metal91.jpg")
chemins_a_exclure.add(chemin_metal91)

print("Nombre total d'images a exclure :", len(chemins_a_exclure))

images_a_nettoyer = []
for ligne in lignes_valides:
    if ligne["chemin"] not in chemins_a_exclure:
        images_a_nettoyer.append(ligne)

print("Nombre d'images restantes a nettoyer :", len(images_a_nettoyer))

Nombre total d'images a exclure : 35
Nombre d'images restantes a nettoyer : 991


# Partie 9 : Redimensionnement

In [51]:
def redimensionner_avec_padding(image, taille_cible=(224, 224), couleur_padding=(0, 0, 0)):
    """
    Redimensionne une image en conservant les proportions,
    puis ajoute du padding pour atteindre exactement taille_cible.
    """
    largeur_cible, hauteur_cible = taille_cible
    largeur_originale, hauteur_originale = image.size

    ratio_largeur = largeur_cible / largeur_originale
    ratio_hauteur = hauteur_cible / hauteur_originale
    ratio = min(ratio_largeur, ratio_hauteur)   # le plus petit ratio pour ne rien deformer

    nouvelle_largeur = int(largeur_originale * ratio)
    nouvelle_hauteur = int(hauteur_originale * ratio)
    image_redimensionnee = image.resize((nouvelle_largeur, nouvelle_hauteur))

    image_finale = Image.new("RGB", taille_cible, couleur_padding)
    position_x = (largeur_cible - nouvelle_largeur) // 2
    position_y = (hauteur_cible - nouvelle_hauteur) // 2
    image_finale.paste(image_redimensionnee, (position_x, position_y))
    return image_finale


# test sur une image reelle du dataset filtre
image_exemple = Image.open(images_a_nettoyer[0]["chemin"])
print("Taille avant :", image_exemple.size)

image_redimensionnee = redimensionner_avec_padding(image_exemple)
print("Taille apres :", image_redimensionnee.size)

Taille avant : (512, 384)
Taille apres : (224, 224)


# Partie 10 : Uniformisation des canaux

In [52]:
def uniformiser_en_rgb(image):
    """Convertit n'importe quel mode (L, RGBA, P, ...) en RGB standard."""
    if image.mode != "RGB":
        image = image.convert("RGB")
    return image


# test sur TOUTES les images de "images_a_nettoyer" (voir etape de filtrage precedente)
nb_convertis = 0
nb_deja_rgb = 0
for ligne in images_a_nettoyer:
    image = Image.open(ligne["chemin"])
    mode_avant = image.mode
    image = uniformiser_en_rgb(image)
    if mode_avant != "RGB":
        nb_convertis += 1
    else:
        nb_deja_rgb += 1

print("Images deja en RGB :", nb_deja_rgb)
print("Images converties en RGB :", nb_convertis)

Images deja en RGB : 979
Images converties en RGB : 12


# Partie 11 : Mise à l'échelle des pixels

In [53]:
def normaliser_pixels(image):
    """Convertit l'image en tableau numpy avec des valeurs entre 0 et 1 (au lieu de 0-255)."""
    tableau = np.array(image).astype("float32")
    tableau_normalise = tableau / 255.0
    return tableau_normalise


# test sur une image reelle
image_exemple = Image.open(images_a_nettoyer[0]["chemin"]).convert("RGB")
tableau_avant = np.array(image_exemple)
print("Avant normalisation -> min:", tableau_avant.min(), " max:", tableau_avant.max(), " dtype:", tableau_avant.dtype)

tableau_normalise = normaliser_pixels(image_exemple)
print("Apres normalisation -> min:", tableau_normalise.min(), " max:", tableau_normalise.max(), " dtype:", tableau_normalise.dtype)

Avant normalisation -> min: 0  max: 253  dtype: uint8
Apres normalisation -> min: 0.0  max: 0.99215686  dtype: float32


# Partie 12 : Découpage Train/Validation/Test

In [54]:
import random

def decouper_dataset(images, proportion_train=0.7, proportion_val=0.15, graine=42):
    """
    Repartit les images de chaque classe en train / validation / test,
    classe par classe, pour garder les proportions dans chaque sous-ensemble.
    """
    random.seed(graine)

    images_par_classe = {}
    for ligne in images:
        classe = ligne["classe"]
        if classe not in images_par_classe:
            images_par_classe[classe] = []
        images_par_classe[classe].append(ligne)

    repartition = {"train": [], "validation": [], "test": []}

    for classe, liste_images in images_par_classe.items():
        random.shuffle(liste_images)

        nb_total = len(liste_images)
        nb_train = int(nb_total * proportion_train)
        nb_val = int(nb_total * proportion_val)

        images_train = liste_images[0:nb_train]
        images_val = liste_images[nb_train:nb_train + nb_val]
        images_test = liste_images[nb_train + nb_val:]

        for ligne in images_train:
            repartition["train"].append(ligne)
        for ligne in images_val:
            repartition["validation"].append(ligne)
        for ligne in images_test:
            repartition["test"].append(ligne)

    return repartition


repartition = decouper_dataset(images_a_nettoyer)
print("Train :", len(repartition["train"]))
print("Validation :", len(repartition["validation"]))
print("Test :", len(repartition["test"]))

Train : 690
Validation : 146
Test : 155


# Partie 13 : Data Augmentation

1) A l’aide de Keras, appliquer à la classe minoritaire de la « data augmentation » en utilisant des
transformations comme rotation légère, retournement horizontal, zoom, translation, variation
de luminosité, variation de contraste, légère variation de couleur, …

In [58]:


def augmenter_classe_minoritaire(images_source, dossier_destination, nb_images_a_generer):
    """
    Genere de nouvelles images (data augmentation) a partir d'une liste
    d'images existantes d'une classe minoritaire, pour equilibrer le dataset.
    """
    generateur = ImageDataGenerator(
        rotation_range=15,           # rotation legere (+/- 15 degres)
        horizontal_flip=True,        # retournement horizontal
        zoom_range=0.1,              # zoom leger
        width_shift_range=0.1,       # translation horizontale
        height_shift_range=0.1,      # translation verticale
        brightness_range=[0.8, 1.2], # variation de luminosite
        channel_shift_range=20       # legere variation de couleur
    )

    os.makedirs(dossier_destination, exist_ok=True)

    nb_generees = 0
    while nb_generees < nb_images_a_generer:
        for ligne in images_source:
            if nb_generees >= nb_images_a_generer:
                break

            image = Image.open(ligne["chemin"]).convert("RGB")
            tableau = np.array(image)
            tableau = tableau.reshape((1,) + tableau.shape)  # forme attendue par le generateur

            flux = generateur.flow(tableau, batch_size=1)
            image_augmentee = next(flux)[0].astype("uint8")

            nom_sortie = "augmentee_" + str(nb_generees) + "_" + ligne["nom"]
            Image.fromarray(image_augmentee).save(os.path.join(dossier_destination, nom_sortie))
            nb_generees += 1


# on applique seulement sur le TRAIN de la classe minoritaire (voir explication plus bas)
images_trash_train = []
for ligne in repartition["train"]:
    if ligne["classe"] == "trash":
        images_trash_train.append(ligne)

print("Images trash disponibles en train :", len(images_trash_train))

# porter trash de 34 a environ 152 (le niveau de plastic en train)
augmenter_classe_minoritaire(
    images_trash_train,
    "../data/cleaned/train/trash",
    nb_images_a_generer=118
)

Images trash disponibles en train : 34
